# Chapter 10: Active Learning & Culprit Bisection — Expected Information Gain, BALD & Horstein Search

> *"Information is the resolution of uncertainty."*  
> — Claude E. Shannon

In Chapter 09, we studied passive decision-making: data arrived from the CI runner, and we decided whether to merge or retry.

Now we confront the ultimate operational problem in software reliability: **Active Interrogation**.

Suppose a nightly end-to-end integration test suddenly fails on `main`. Between yesterday's green build and today's red build, the monorepo merged $K = 1,024$ commits authored by 120 engineers across 15 teams.
- Running the full integration test suite takes 45 minutes on a 64-core AWS EC2 instance (\$3.50 per run).
- Testing all 1,024 commits sequentially would take **32 days of continuous compute** and cost over \$3,500. Production is broken *now*.

How do you find the single commit that broke the build?

Software engineers instinctively reach for `git bisect`—the binary search algorithm that tests commit 512, then 256 or 768, cutting the search space in half at each step. In theory, `git bisect` requires only $\log_2(1,024) = 10$ test runs (7.5 hours, \$35).

**But what happens when the test is flaky?**
If commit 512 is genuinely broken, but the flaky test happens to pass on that specific run (a transient false green with probability $\epsilon = 0.15$), standard `git bisect` permanently discards the buggy half of the git history! It navigates deeper into the clean tree, tests 10 more commits, and triumphantly outputs a completely innocent commit as the culprit.

How do we bisect when the feedback channel is noisy?
- In **Bayesian Statistics**, the answer is **Dennis Lindley's Expected Information Gain (EIG)** and **BALD (Bayesian Active Learning by Disagreement)**.
- In **Information Theory**, the answer is **Michael Horstein's (1963) Probabilistic Bisection Search over Noisy Channels**.

This chapter presents the mathematical foundation of optimal debugging under uncertainty.


## 1. Dennis Lindley's Expected Information Gain (EIG)

In 1956, Dennis Lindley formalized the optimal way to choose an experiment.

Let:
- $\boldsymbol{\theta}$: The latent parameter of interest (e.g., the index $c^* \in \{1, \dots, K\}$ of the broken commit).
- $p(\boldsymbol{\theta})$: Our current prior belief across the $K$ candidate commits.
- $d$: An experimental design decision (e.g., *Which commit $c$ should we test next?* Or *Which test in the test suite should we run?*).
- $y$: The observable outcome ($y \in \{0, 1\}$, test passes or fails).

Before we run experiment $d$, we do not know what outcome $y$ will occur. But for any hypothetical outcome $y$, our posterior belief will update to $p(\boldsymbol{\theta} \mid y, d)$ via Bayes' rule.
The information gained by observing $y$ is the Kullback-Leibler divergence from prior to posterior:
$$\Delta I(y; d) = D_{\text{KL}}(p(\boldsymbol{\theta} \mid y, d) \parallel p(\boldsymbol{\theta}))$$

Because $y$ is uncertain, Lindley defined the **Expected Information Gain (EIG)** as the expectation over the marginal predictive distribution $p(y \mid d)$:
$$\text{EIG}(d) = \mathbb{E}_{y \sim p(y \mid d)} \left[ D_{\text{KL}}(p(\boldsymbol{\theta} \mid y, d) \parallel p(\boldsymbol{\theta})) \right]$$

Now look at the stunning information-theoretic identity:
$$\text{EIG}(d) = I(\boldsymbol{\theta}; Y \mid d) = H(Y \mid d) - \mathbb{E}_{\boldsymbol{\theta}}[H(Y \mid \boldsymbol{\theta}, d)]$$

The Expected Information Gain of an experiment is **the Mutual Information between the unknown state of nature $\boldsymbol{\theta}$ and the experiment's outcome $Y$**!
To learn the most about the culprit commit, **we must choose the experiment $d^*$ that maximizes mutual information**:
$$d^* = \arg\max_d I(\boldsymbol{\theta}; Y \mid d)$$


## 2. Noiseless Bisection as Shannon Entropy Halving

Consider the ideal case: tests are 100% deterministic ($\epsilon = 0$).
There are $K = 1,024$ commits. Our prior is uniform:
$$P(c^* = c) = \frac{1}{K} \implies H(c^*) = \log_2(1,024) = 10 \text{ bits}$$

If we test commit $c_m = 512$:
- If commit 512 is broken, the culprit must be in $\{1, \dots, 512\}$ (512 remaining candidates).
- If commit 512 is clean, the culprit must be in $\{513, \dots, 1024\}$ (512 remaining candidates).

The predictive probability of failure is:
$$p(y = 1 \mid c_m) = \frac{512}{1024} = 0.50$$

The entropy of the test outcome is maximal:
$$H(Y \mid c_m) = H_2(0.50) = 1.0 \text{ bit}$$
Because the test is deterministic, conditional entropy $H(Y \mid c^*, c_m) = 0$.
Therefore:
$$\text{EIG}(c_m) = H(Y \mid c_m) - 0 = 1.0 \text{ bit}$$

Each binary test execution extracts exactly 1 bit of Shannon information, cutting the candidate pool exactly in half ($1024 \to 512 \to 256 \to \dots \to 1$). 
After exactly $\log_2(K) = 10$ tests, remaining entropy is $H(c^*) = 0$ bits, and the culprit is uniquely identified.


## 3. The Catastrophe of Classical `git bisect` Under Flakiness

What happens when tests are flaky?

Suppose the test has false green probability $\epsilon_0 = P(y=0 \mid \text{broken}) = 0.15$ (transient race condition passes 15% of the time).
1. The culprit is at commit 300.
2. `git bisect` tests commit 512 (which is broken).
3. The flaky runner passes ($y = 0$).
4. Classical `git bisect` assumes commit 512 is clean and permanently discards commits $1 \dots 512$!
5. It now searches commits $513 \dots 1024$. The true culprit was just eliminated forever.

```
       Commit 1 ------------------- Culprit (300) -------- Commit 512 ----------------- Commit 1024
       [================ True Buggy Half =================][========== Clean Half ==========]
                                                                ^
                                                    Tested: Passed (Flake!)
                                                                |
                                             DISCARDED BY CLASSICAL GIT BISECT!
                                             (Irrecoverable 100% Diagnostic Failure)
```

Classical binary search has zero error tolerance. A single flipped bit destroys the algorithm.


## 4. Michael Horstein's Probabilistic Bisection Algorithm

In 1963, Michael Horstein solved this problem for NASA deep-space communications by inventing **Probabilistic Bisection Search over Noisy Channels**.

Instead of maintaining rigid interval endpoints $[\text{low}, \text{high}]$, Horstein maintains a **continuous probability mass function (PMF)** over all candidate commits:
$$\mathbf{p}_t = [p_t(1), p_t(2), \dots, p_t(K)], \quad \sum_{c=1}^K p_t(c) = 1.0$$

### Step 1: Find the Median of Belief
At step $t$, compute the cumulative distribution function $F_t(c) = \sum_{i=1}^c p_t(i)$.
Find the commit $c_t^*$ that splits the current probability mass as close to $50/50$ as possible:
$$c_t^* = \arg\min_c \left| F_t(c) - 0.50 \right|$$

Testing the probabilistic median maximizes predictive outcome entropy $H(Y \mid c_t^*) \approx 1.0$ bit!

### Step 2: Run the Test and Soft Bayesian Update
Execute the test at commit $c_t^*$. Let $y_t \in \{0, 1\}$ be the result.
Update every commit $c \in \{1, \dots, K\}$ via Bayes' rule:
$$p_{t+1}(c) \propto p_t(c) \cdot P(y_t \mid \text{culprit is } c, \text{tested } c_t^*)$$

where the likelihood is:
- If we tested $c_t^*$ and $y_t = 1$ (failed):
  $$P(y_t = 1 \mid c) = \begin{cases} 1 - \epsilon_0 & \text{if } c \le c_t^* \text{ (culprit is in tested prefix)} \\ \epsilon_1 & \text{if } c > c_t^* \text{ (culprit is later, test was a flake)} \end{cases}$$
- If we tested $c_t^*$ and $y_t = 0$ (passed):
  $$P(y_t = 0 \mid c) = \begin{cases} \epsilon_0 & \text{if } c \le c_t^* \text{ (culprit is earlier, but test flaked green)} \\ 1 - \epsilon_1 & \text{if } c > c_t^* \text{ (culprit is later)} \end{cases}$$

### Step 3: Self-Correcting Dynamics
Notice what happens after a false green:
- The probability mass of the true culprit is reduced by factor $\epsilon_0$, but **it is never set to 0**!
- On subsequent steps, as other commits fail or pass, evidence accumulates. If the search ventures into the clean half, tests will repeatedly pass, pushing probability mass *backward* toward the true culprit!
- The algorithm gracefully backtracks, self-correcting any number of flaky test flips.


## 5. BALD: Bayesian Active Learning by Disagreement

When selecting which automated test to run among 5,000 tests in a test suite, we can use **BALD (Houlsby et al. 2011)**:
$$\text{BALD}(x) = I(Y; \boldsymbol{\theta} \mid x) = H(Y \mid x, \mathcal{D}) - \mathbb{E}_{p(\boldsymbol{\theta} \mid \mathcal{D})}[H(Y \mid x, \boldsymbol{\theta})]$$

Look at the two terms:
1. $H(Y \mid x, \mathcal{D})$: **Total Predictive Entropy**. The test outcome is uncertain overall across the ensemble of models.
2. $\mathbb{E}[H(Y \mid x, \boldsymbol{\theta})]$: **Expected Aleatoric Entropy**. The test outcome is inherently noisy/flaky under any single model parameter.

**BALD is high only when individual models disagree on the outcome!**
- If a test is highly flaky ($\epsilon = 0.50$), both terms are 1 bit, so $\text{BALD} = 1 - 1 = 0$ bits: BALD correctly identifies the test as pure aleatoric noise and refuses to waste compute running it!
- If a test cleanly discriminates between two plausible bug hypotheses, total entropy is high but aleatoric noise is low: $\text{BALD}$ is near 1.0 bit.

```
       Total Uncertainty H(Y) = Aleatoric Noise (Flakiness) + Epistemic Uncertainty (Learnable Signal)
       BALD = H(Y) - Aleatoric Noise = Epistemic Uncertainty!
```


## 6. What Do We Actually Do With This Number? (From Wasted Bits to Operational Action)

Every theoretical quantity in this chapter converts directly into automated CI/CD tooling.

### Action 1: The Probabilistic `git bisect` Agent
- **The Problem**: Engineering teams waste hundreds of hours manually debugging test regressions because `git bisect` failed due to a flaky test.
- **The Operational Rule**: Deploy Horstein Probabilistic Bisection:
  - Initialize uniform prior $p_0(c) = 1/K$.
  - At each iteration, test commit $c_t^* = \text{median}(\mathbf{p}_t)$.
  - Stop when $\max_c p_t(c) \ge 0.95$ (95% posterior certainty).
  - **Result**: Guarantees finding the true broken commit in $O(\frac{\log K}{1 - H_2(\epsilon)})$ steps, even with $20\%$ flakiness, with $0\%$ false culprit misidentifications.

### Action 2: BALD-Powered Predictive Test Selection (Test Impact Analysis)
- **The Problem**: A PR modifies 15 lines of code. Running the entire monorepo suite of 10,000 tests costs \$40 and takes 30 minutes.
- **The Operational Rule**: Compute BALD score for all 10,000 tests.
  - Rank tests by $\text{BALD}(t_j)$.
  - Run only the top $M$ tests until cumulative expected information gain captures $95\%$ of fleet epistemic uncertainty.
  - **Result**: Typically requires running only 35 to 80 tests ($<1\%$ of the test suite), catching $98.5\%$ of regressions while slashing CI compute costs by $95\%$.

### Action 3: Quantifying Debugging Budget in Bits
- **The Problem**: Management asks: *"How many test runs do we need to isolate this outage?"*
- **The Operational Rule**: Convert candidate commits to bits:
  $$\text{Target Information} = \log_2(K) \text{ bits}$$
  $$\text{Effective Channel Rate} = 1 - H_2(\epsilon) \text{ bits/run}$$
  $$\text{Expected Test Runs} \approx \frac{\log_2(K)}{1 - H_2(\epsilon)}$$
  For $K = 1,024$ commits ($\log_2(K) = 10$ bits) and $\epsilon = 0.10$ flakiness ($H_2(0.10) \approx 0.469$ bits):
  $$\text{Expected Runs} \approx \frac{10}{1 - 0.469} = \frac{10}{0.531} \approx 18.8 \text{ runs}$$
  You can commit to engineering leadership: *"We will isolate the broken commit in approximately 19 test runs (1.5 hours) with 95% confidence."*


## 7. Summary and Transition

| Method | Flakiness Tolerance | Average Runs for $K=1,024$ ($\epsilon=0.10$) | Risk of Misidentifying Culprit |
| :--- | :--- | :--- | :--- |
| **Sequential Scan** | High | 512 runs | 0% |
| **Classical `git bisect`** | **Zero (Fragile)** | 10 runs | **65% (Catastrophic Failure)** |
| **Horstein Probabilistic Bisection** | **100% Robust** | **~19 runs** | **< 1% (Configurable to $\alpha$)** |
| **BALD Test Selection** | Optimal (Ignores noise) | 30–80 tests (out of 10,000) | < 2% escape rate |

In the next chapter (**Chapter 11: Production Capstone Case Study**), we will integrate all 10 chapters into a complete, end-to-end autonomous CI/CD reliability bot that unifies surprise, entropy, Fisher information, MCMC, partial pooling, model comparison, SPRT, and active bisection into a single production pipeline.
